# Phase 7 — Predictive Modeling Strategy & Target Feasibility
## Australian Stock Market Analytics

### Purpose
Before training a predictive model, define what is being predicted, whether the current data supports that target, and what additional data is required.

The current workbook contains 53 companies with financial metrics measured at one cross-sectional observation. It is suitable for descriptive analysis and exploratory company segmentation, but it does **not** contain repeated dated observations or future outcomes. Therefore, this phase will **not** train a fake stock-price forecasting model or invent target labels.

This notebook will:
1. load and validate the Phase 3 feature-engineered dataset;
2. inventory available columns and identify date/time and target candidates;
3. document feasible prediction tasks versus unsupported tasks;
4. check whether the current data can support a valid supervised-learning target;
5. create a data acquisition specification for a future predictive phase;
6. save a phase report and target/data requirements.

**Phase 7 deliverable:** a documented, defensible modeling plan. Model training begins only after a valid target and suitable observations are available.

## 1. Imports

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Imports successful.")

Imports successful.


## 2. Locate and load the Phase 3 dataset

In [2]:
INPUT_FILENAME = "australian_stocks_feature_engineered.csv"
PROJECT_ROOT = None
input_file = None

for base in [Path.cwd(), *Path.cwd().parents]:
    candidate = base / "data" / "processed" / INPUT_FILENAME
    if candidate.exists():
        PROJECT_ROOT, input_file = base, candidate
        break

if input_file is None:
    raise FileNotFoundError(
        f"Cannot find {INPUT_FILENAME}. Make sure Phase 3 saved it in data/processed/."
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORT_DIR = PROJECT_ROOT / "reports" / "predictive_modeling_strategy"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(input_file)
print("Input file:", input_file)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Unique tickers:", df["ticker"].nunique() if "ticker" in df.columns else "ticker column missing")
display(df.head())

Input file: e:\Projects\Australian Stock Market Data Science Project\data\processed\australian_stocks_feature_engineered.csv
Rows: 53
Columns: 47
Unique tickers: 53


,ticker,company_name,total_shares,market_cap_m,eps_2h2024,eps_1h2025,eps_ltm,eps_ap_24_25,pe_annual_report,pe_ltm,price,book_value_per_share,week52_range,duplicate_ticker_flag,annual_pe_missing,week52_low,week52_high,market_cap_m_original,eps_ltm_original,pe_ltm_original,total_shares_outlier,market_cap_m_outlier,eps_ltm_outlier,eps_ap_24_25_outlier,pe_ltm_outlier,price_outlier,book_value_per_share_outlier,week52_range_valid,log_market_cap_m,market_cap_size_band,eps_half_year_difference,eps_ltm_vs_annual_difference,eps_ltm_vs_annual_pct_difference,earnings_yield_pct,price_to_book,pe_ltm_minus_annual,pe_ltm_vs_annual_pct_difference,week52_range_width,price_position_52w,price_distance_from_52w_low_pct,price_distance_from_52w_high_pct,price_outside_52w_range,annual_pe_missing_flag,negative_ltm_eps_flag,negative_ltm_pe_flag,nonpositive_book_value_flag,invalid_52w_range_flag
0,VEA,Viva Energy Group Ltd,1625651926,"3,511.4082",-0.3000,12.3000,12.0000,4.8000,450.0000,18.0000,2.1600,1.0270,1.41 - 2.83,False,False,1.4100,2.8300,"3,511.4082",12.0000,18.0000,False,False,False,False,False,False,False,True,8.1638,Medium (sample-relative),12.6000,7.2000,150.0000,5.5556,2.1032,-432.0000,-96.0000,1.4200,0.5282,53.1915,23.6749,False,False,False,False,False,False
1,DDR,Dicker Data Ltd,180817855,"1,871.4648",23.9700,21.7900,45.7600,43.6200,237.2765,22.6180,10.3500,1.3890,7.58 - 10.94,False,False,7.5800,10.9400,"1,871.4648",45.7600,22.6180,False,False,False,False,False,False,False,True,7.5345,Medium (sample-relative),-2.1800,2.1400,4.9060,4.4213,7.4514,-214.6585,-90.4677,3.3600,0.8244,36.5435,5.3931,False,False,False,False,False,False
2,FMG,Fortescue Metals Group Ltd,3078964918,"68,075.9143",50.5000,39.5500,90.0500,109.7000,201.5497,24.5530,22.1100,9.8740,13.18 - 22.11,False,False,13.1800,22.1100,"68,075.9143",90.0500,24.5530,True,True,False,False,False,False,False,True,11.1284,Large (sample-relative),-10.9500,-19.6500,-17.9125,4.0728,2.2392,-176.9967,-87.8179,8.9300,1.0000,67.7542,0.0000,False,False,False,False,False,False
3,RIO,RIO Tinto Ltd,371216214,"51,402.3092",315.2000,296.0000,611.2000,669.5000,206.8260,22.6554,138.4700,54.4740,100.75 - 140.58,False,False,100.7500,140.5800,"51,402.3092",611.2000,22.6554,False,True,True,True,False,True,True,True,10.8474,Large (sample-relative),-19.2000,-58.3000,-8.7080,4.4140,2.5419,-184.1706,-89.0461,39.8300,0.9470,37.4392,1.5009,False,False,False,False,False,False
4,APE,Eagers Automotive Ltd,282364338,"7,578.6588",34.8000,46.4000,81.2000,80.2000,334.6633,33.0542,26.8400,4.6360,10.90 - 35.64,False,False,10.9000,35.6400,"7,578.6588",81.2000,33.0542,False,False,False,False,False,False,False,True,8.9331,Medium (sample-relative),11.6000,1.0000,1.2469,3.0253,5.7895,-301.6092,-90.1232,24.7400,0.6443,146.2385,24.6914,False,False,False,False,False,False


## 3. Validate the dataset and inventory its columns

In [3]:
if "ticker" not in df.columns:
    raise ValueError("Expected identifier column 'ticker' is missing.")

inventory = pd.DataFrame({
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [int(df[c].isna().sum()) for c in df.columns],
    "missing_pct": [float(df[c].isna().mean() * 100) for c in df.columns],
    "unique_values": [int(df[c].nunique(dropna=True)) for c in df.columns]
})

display(inventory)
inventory.to_csv(REPORT_DIR / "available_column_inventory.csv", index=False)

validation = pd.DataFrame([{
    "rows": len(df),
    "columns": len(df.columns),
    "unique_tickers": int(df["ticker"].nunique()),
    "duplicate_tickers": int(df["ticker"].duplicated().sum()),
    "has_date_column": any(
        token in c.lower() for c in df.columns
        for token in ["date", "timestamp", "year", "month", "period"]
    ),
    "has_explicit_future_target": any(
        token in c.lower() for c in df.columns
        for token in ["future_return", "next_return", "target", "label", "future_price"]
    )
}])
display(validation)
validation.to_csv(REPORT_DIR / "dataset_validation.csv", index=False)

,column,dtype,missing_count,missing_pct,unique_values
0,ticker,str,0,0.0000,53
1,company_name,str,0,0.0000,53
2,total_shares,int64,0,0.0000,53
3,market_cap_m,float64,0,0.0000,53
4,eps_2h2024,float64,0,0.0000,53
5,eps_1h2025,float64,0,0.0000,53
6,eps_ltm,float64,0,0.0000,53
7,eps_ap_24_25,float64,0,0.0000,52
8,pe_annual_report,float64,2,3.7736,51
9,pe_ltm,float64,0,0.0000,53


,rows,columns,unique_tickers,duplicate_tickers,has_date_column,has_explicit_future_target
0,53,47,53,0,True,False


## 4. Identify likely identifiers, current features, and possible targets
Column names alone cannot establish that a variable is a valid target. A target must be defined with a clear business question, timing, and a label/value that is observed independently of the predictors.

In [4]:
identifier_candidates = [
    c for c in df.columns
    if c.lower() in {"ticker", "company_name", "company", "name", "symbol", "id"}
]
date_candidates = [
    c for c in df.columns
    if any(token in c.lower() for token in ["date", "timestamp", "year", "month", "period"])
]
target_name_candidates = [
    c for c in df.columns
    if any(token in c.lower() for token in [
        "target", "label", "future", "next_return", "future_return",
        "future_price", "next_price", "outcome"
    ])
]

print("Identifier candidates:", identifier_candidates)
print("Date/time candidates:", date_candidates)
print("Possible target-named columns:", target_name_candidates)

candidate_review = pd.DataFrame([
    {"category": "Identifier", "column": c, "status": "Exclude from model inputs"}
    for c in identifier_candidates
] + [
    {"category": "Date/time", "column": c, "status": "No date column detected" if not date_candidates else "Review meaning and granularity"}
    for c in (date_candidates or ["<none detected>"])
] + [
    {"category": "Target name", "column": c, "status": "No explicit future target detected" if not target_name_candidates else "Inspect definition and leakage risk"}
    for c in (target_name_candidates or ["<none detected>"])
])
display(candidate_review)
candidate_review.to_csv(REPORT_DIR / "candidate_column_review.csv", index=False)

Identifier candidates: ['ticker', 'company_name']
Date/time candidates: ['eps_half_year_difference']
Possible target-named columns: []


,category,column,status
0,Identifier,ticker,Exclude from model inputs
1,Identifier,company_name,Exclude from model inputs
2,Date/time,eps_half_year_difference,Review meaning and granularity
3,Target name,<none detected>,No explicit future target detected


## 5. Is genuine stock-price forecasting possible with the current data?

In [5]:
# This is a structural feasibility check, not a model-performance test.
date_like_cols = [
    c for c in df.columns
    if any(token in c.lower() for token in ["date", "timestamp", "trading_day", "observation_date"])
]
price_like_cols = [
    c for c in df.columns
    if any(token in c.lower() for token in ["price", "close", "adjusted_close", "adj_close"])
]
future_target_cols = [
    c for c in df.columns
    if any(token in c.lower() for token in [
        "future_return", "next_return", "future_price", "next_price", "target"
    ])
]

forecasting_feasibility = {
    "has_dated_observations": bool(date_like_cols),
    "date_columns": date_like_cols,
    "price_like_columns": price_like_cols,
    "explicit_future_target_columns": future_target_cols,
    "has_repeated_time_series_structure": False,
    "genuine_future_price_forecasting_supported": False,
    "reason": (
        "The supplied dataset is cross-sectional and does not contain repeated dated "
        "prices or an observed future outcome. A forecast target cannot be derived "
        "validly from a single snapshot."
    )
}
display(pd.DataFrame([forecasting_feasibility]))
with open(REPORT_DIR / "forecasting_feasibility.json", "w", encoding="utf-8") as f:
    json.dump(forecasting_feasibility, f, indent=2)

,has_dated_observations,date_columns,price_like_columns,explicit_future_target_columns,has_repeated_time_series_structure,genuine_future_price_forecasting_supported,reason
0,False,[],"[price, price_outlier, price_to_book, price_position_52w, price_distance_from_52w_low_pct, price_distance_from_52w_h...",[],False,False,The supplied dataset is cross-sectional and does not contain repeated dated prices or an observed future outcome. A ...


## 6. Candidate prediction tasks and whether the current data supports them
The distinction below prevents us from training a model merely because a library is available. Some tasks are possible only if a valid label is added or independently verified.

In [6]:
task_options = pd.DataFrame([
    {
        "task": "Forecast future stock price or return",
        "target_example": "Future adjusted close or forward return over a defined horizon",
        "current_data_status": "NOT SUPPORTED",
        "additional_requirement": "Dated historical adjusted prices per ticker, consistent date range, and a defined forecast horizon",
        "notes": "Requires time-aware train/validation/test splits; avoid random row splits across time."
    },
    {
        "task": "Classify future outperformance",
        "target_example": "Whether a stock beats a defined benchmark over the next 3 or 6 months",
        "current_data_status": "NOT SUPPORTED",
        "additional_requirement": "Future stock returns and matching benchmark returns for the same horizon",
        "notes": "Define benchmark and label threshold before examining model results."
    },
    {
        "task": "Predict current valuation metric",
        "target_example": "Current P/E or price-to-book",
        "current_data_status": "TECHNICALLY POSSIBLE, BUT NOT RECOMMENDED AS A PREDICTION GOAL",
        "additional_requirement": "A defensible independent target and predictors available at prediction time",
        "notes": "Using related accounting/price features can create target leakage or a circular task."
    },
    {
        "task": "Explain or segment current company profiles",
        "target_example": "Descriptive groups based on observed financial features",
        "current_data_status": "SUPPORTED AS EXPLORATORY ANALYSIS",
        "additional_requirement": "No additional data for descriptive segmentation",
        "notes": "Already explored in Phase 6; clusters are not future outcomes."
    },
    {
        "task": "Predict financial distress or default",
        "target_example": "A clearly defined future distress/default event",
        "current_data_status": "NOT SUPPORTED",
        "additional_requirement": "Verified historical outcome labels, dates, and sufficient positive cases",
        "notes": "A label cannot be inferred from one negative EPS or P/E observation."
    }
])
display(task_options)
task_options.to_csv(REPORT_DIR / "candidate_prediction_tasks.csv", index=False)

,task,target_example,current_data_status,additional_requirement,notes
0,Forecast future stock price or return,Future adjusted close or forward return over a defined horizon,NOT SUPPORTED,"Dated historical adjusted prices per ticker, consistent date range, and a defined forecast horizon",Requires time-aware train/validation/test splits; avoid random row splits across time.
1,Classify future outperformance,Whether a stock beats a defined benchmark over the next 3 or 6 months,NOT SUPPORTED,Future stock returns and matching benchmark returns for the same horizon,Define benchmark and label threshold before examining model results.
2,Predict current valuation metric,Current P/E or price-to-book,"TECHNICALLY POSSIBLE, BUT NOT RECOMMENDED AS A PREDICTION GOAL",A defensible independent target and predictors available at prediction time,Using related accounting/price features can create target leakage or a circular task.
3,Explain or segment current company profiles,Descriptive groups based on observed financial features,SUPPORTED AS EXPLORATORY ANALYSIS,No additional data for descriptive segmentation,Already explored in Phase 6; clusters are not future outcomes.
4,Predict financial distress or default,A clearly defined future distress/default event,NOT SUPPORTED,"Verified historical outcome labels, dates, and sufficient positive cases",A label cannot be inferred from one negative EPS or P/E observation.


## 7. Data requirements for a defensible future forecasting phase
A practical next step is to acquire dated market data and, if possible, point-in-time fundamentals. Keep data timestamps aligned so features only use information available at the prediction date.

In [7]:
data_requirements = pd.DataFrame([
    {
        "dataset": "Daily adjusted market prices",
        "required_fields": "ticker, date, adjusted_close (or documented equivalent), currency",
        "purpose": "Calculate historical returns and define forward-return targets",
        "priority": "Required for price/return forecasting",
        "quality_checks": "Trading dates, duplicate ticker-date pairs, missing periods, splits/dividends treatment, delisted tickers"
    },
    {
        "dataset": "Benchmark index",
        "required_fields": "benchmark ticker, date, adjusted_close",
        "purpose": "Define relative performance/outperformance labels",
        "priority": "Required for benchmark-relative classification",
        "quality_checks": "Same date range and compatible return calculation"
    },
    {
        "dataset": "Point-in-time financial fundamentals",
        "required_fields": "ticker, report_period, publication_date, fundamental metrics",
        "purpose": "Build predictors without look-ahead leakage",
        "priority": "Recommended for fundamental forecasting",
        "quality_checks": "Use publication date, not only accounting period end; track restatements"
    },
    {
        "dataset": "Company/ticker mapping",
        "required_fields": "ticker, company identifier, effective start/end dates",
        "purpose": "Handle ticker changes and survivorship issues",
        "priority": "Recommended",
        "quality_checks": "Corporate actions, renamed/delisted firms, mapping validity"
    }
])
display(data_requirements)
data_requirements.to_csv(REPORT_DIR / "future_data_requirements.csv", index=False)

,dataset,required_fields,purpose,priority,quality_checks
0,Daily adjusted market prices,"ticker, date, adjusted_close (or documented equivalent), currency",Calculate historical returns and define forward-return targets,Required for price/return forecasting,"Trading dates, duplicate ticker-date pairs, missing periods, splits/dividends treatment, delisted tickers"
1,Benchmark index,"benchmark ticker, date, adjusted_close",Define relative performance/outperformance labels,Required for benchmark-relative classification,Same date range and compatible return calculation
2,Point-in-time financial fundamentals,"ticker, report_period, publication_date, fundamental metrics",Build predictors without look-ahead leakage,Recommended for fundamental forecasting,"Use publication date, not only accounting period end; track restatements"
3,Company/ticker mapping,"ticker, company identifier, effective start/end dates",Handle ticker changes and survivorship issues,Recommended,"Corporate actions, renamed/delisted firms, mapping validity"


## 8. Proposed target definition template
Do not train a model until the target definition is explicit. Fill this template after selecting the task and obtaining suitable data.

In [8]:
target_template = pd.DataFrame([
    {"item": "Prediction question", "decision": "To be decided"},
    {"item": "Target variable", "decision": "Not defined yet"},
    {"item": "Forecast horizon", "decision": "Not defined yet (e.g., 1, 3, or 6 months)"},
    {"item": "Prediction timestamp", "decision": "Not defined yet"},
    {"item": "Universe", "decision": "Companies included and eligibility rules to be documented"},
    {"item": "Benchmark", "decision": "Required only for relative-return tasks"},
    {"item": "Feature availability rule", "decision": "Every predictor must be available on or before prediction timestamp"},
    {"item": "Validation strategy", "decision": "Chronological split / walk-forward validation for time-dependent targets"},
    {"item": "Evaluation metrics", "decision": "Choose only after target type and business objective are fixed"}
])
display(target_template)
target_template.to_csv(REPORT_DIR / "target_definition_template.csv", index=False)

,item,decision
0,Prediction question,To be decided
1,Target variable,Not defined yet
2,Forecast horizon,"Not defined yet (e.g., 1, 3, or 6 months)"
3,Prediction timestamp,Not defined yet
4,Universe,Companies included and eligibility rules to be documented
5,Benchmark,Required only for relative-return tasks
6,Feature availability rule,Every predictor must be available on or before prediction timestamp
7,Validation strategy,Chronological split / walk-forward validation for time-dependent targets
8,Evaluation metrics,Choose only after target type and business objective are fixed


## 9. Final phase decision and saved report

In [9]:
phase_decision = {
    "phase": 7,
    "title": "Predictive Modeling Strategy & Target Feasibility",
    "status": "STRATEGY COMPLETED; MODEL TRAINING BLOCKED UNTIL VALID TARGET DATA IS AVAILABLE",
    "dataset_rows": int(len(df)),
    "unique_tickers": int(df["ticker"].nunique()),
    "date_columns_detected": date_candidates,
    "explicit_target_columns_detected": target_name_candidates,
    "forecasting_supported_by_current_snapshot": False,
    "recommended_next_action": (
        "Choose a prediction question, acquire dated historical adjusted prices "
        "and benchmark data if forecasting returns, then define the target and "
        "time-aware validation protocol before training."
    )
}

display(pd.DataFrame([phase_decision]))
with open(REPORT_DIR / "phase7_decision.json", "w", encoding="utf-8") as f:
    json.dump(phase_decision, f, indent=2)

print("Phase 7 reports saved to:", REPORT_DIR)
print("PHASE 7 STRATEGY COMPLETED.")
print("No predictive model was trained because the current snapshot has no valid future target.")

,phase,title,status,dataset_rows,unique_tickers,date_columns_detected,explicit_target_columns_detected,forecasting_supported_by_current_snapshot,recommended_next_action
0,7,Predictive Modeling Strategy & Target Feasibility,STRATEGY COMPLETED; MODEL TRAINING BLOCKED UNTIL VALID TARGET DATA IS AVAILABLE,53,53,[eps_half_year_difference],[],False,"Choose a prediction question, acquire dated historical adjusted prices and benchmark data if forecasting returns, th..."


Phase 7 reports saved to: e:\Projects\Australian Stock Market Data Science Project\reports\predictive_modeling_strategy
PHASE 7 STRATEGY COMPLETED.
No predictive model was trained because the current snapshot has no valid future target.
